In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("../output_csv")

input_file = folder / "residential_cleaned_preprocessed.csv"

df = pd.read_csv(input_file, low_memory=False)

df.head()

,ClosePrice,source_month,LivingArea,DaysOnMarket,LotSizeSquareFeet,YearBuilt,BathroomsTotalInteger,BedroomsTotal,GarageSpaces,Latitude,...,PostalCode_92563,PostalCode_92584,PostalCode_92592,PostalCode_92596,PostalCode_93065,PostalCode_93535,PostalCode_93536,PostalCode_93551,PostalCode_94513,PostalCode_Other
0,1800000.0,202505,1.448510,-0.753592,-0.020607,0.984604,1.229861,1.602918,0.306167,-0.475161,...,False,False,False,False,False,False,False,False,False,True
1,1200000.0,202505,-0.853308,-0.753592,-0.020792,-1.299200,-0.561218,-1.572704,-0.001975,-0.363308,...,False,False,False,False,False,False,False,False,False,True
2,2250000.0,202505,-0.880343,-0.753592,-0.020726,-0.755437,-1.456758,-0.514163,-0.310118,1.467452,...,False,False,False,False,False,False,False,False,False,True
3,1425000.0,202505,-0.360889,-0.753592,-0.020622,0.440841,-0.561218,-0.514163,-0.001975,-0.483640,...,False,False,False,False,False,False,False,False,False,True
4,660000.0,202505,-0.733583,-0.753592,-0.020800,0.368339,0.334321,-0.514163,-0.001975,-0.890424,...,False,False,False,False,False,False,False,False,False,True


In [2]:
target = 'ClosePrice'
# Train/test split by most recent month
df['source_month'] = df['source_month'].astype(float).astype(int).astype(str)

months = sorted(df['source_month'].unique())

test_month = months[-1]

print("Available months:", months)
print("Test month:", test_month)

X_window = 6

train_months = months[-(X_window + 1):-1]

train_df = df[df['source_month'].isin(train_months)].copy()
test_df = df[df['source_month'] == test_month].copy()

print("Training months:", train_months)
print("Testing month:", test_month)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

drop_cols = [target, 'source_month']
feature_cols = [col for col in df.columns if col not in drop_cols]

results = []

for X_window in [3, 6, 9, 12]:
    train_months = months[-(X_window + 1):-1]
    test_month = months[-1]

    train_df = df[df['source_month'].isin(train_months)].copy()
    test_df = df[df['source_month'] == test_month].copy()

    X_train = train_df[feature_cols]
    y_train = train_df[target]

    X_test = test_df[feature_cols]
    y_test = test_df[target]

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append({
        'training_window_months': X_window,
        'train_months': ', '.join(train_months),
        'test_month': test_month,
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    })

results_df = pd.DataFrame(results)
results_df.sort_values('r2')

Available months: ['202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604', '202605']
Test month: 202605
Training months: ['202511', '202512', '202601', '202602', '202603', '202604']
Testing month: 202605
Train shape: (59156, 202)
Test shape: (11973, 202)


,training_window_months,train_months,test_month,mae,rmse,r2
0,3,"202602, 202603, 202604",202605,625859.300317,1.450686e+06,0.253278
1,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,553658.476867,1.370468e+06,0.333577
2,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,531346.068076,1.352861e+06,0.350591
3,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,537784.982295,1.343528e+06,0.359520


In [3]:
output_file = folder / "baseline_model_results.csv"

results_df.to_csv(
    output_file,
    
    index=False
)

best_baseline = results_df.iloc[0]

print("Baseline Model: Linear Regression")
print("Best training window:", best_baseline["training_window_months"], "months")
print("Training months:", best_baseline["train_months"])
print("Test month:", best_baseline["test_month"])
print(f"Test R²: {best_baseline['r2']:.4f}")
print(f"Test MAE: ${best_baseline['mae']:,.2f}")
print(f"Test RMSE: ${best_baseline['rmse']:,.2f}")

Baseline Model: Linear Regression
Best training window: 3 months
Training months: 202602, 202603, 202604
Test month: 202605
Test R²: 0.2533
Test MAE: $625,859.30
Test RMSE: $1,450,685.83
